# DB2Model — ablation

Обучает по адаптеру на каждый вариант данных и выгружает предсказания для каждого.
Всё в одном прогоне: модель, вопросы и код общие, меняются только обучающие данные.

| Вариант | Что проверяет |
|---|---|
| `synth_50`, `synth_143`, `synth_347` | сколько синтетики реально нужно |
| `unfiltered_347` | что даёт фильтр по исполнению (там 24% битого SQL) |
| `real_143` | насколько синтетика хуже живых пар из BIRD train |

`real_143` и `synth_143` одного размера намеренно — иначе качество перепутается с количеством.

**Перед запуском:** GPU + Internet, залей `db2model/kaggle_input/` как датасет, поправь `DATA_DIR`.

Прогон длинный (~1.5 часа): 5 обучений и 5 проходов генерации по 91 вопросу.

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate trl datasets

In [ ]:
import gc
import json
import re
from dataclasses import fields
from pathlib import Path

import torch

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"

# ПОПРАВЬ ПОД СВОЙ ДАТАСЕТ
DATA_DIR = Path("/kaggle/input/datasets/vorange/db2-model-dls")
OUT_DIR = Path("/kaggle/working")

VARIANTS = ["synth_50", "synth_143", "synth_347", "unfiltered_347", "real_143"]

val_pairs = json.loads((DATA_DIR / "val.json").read_text(encoding="utf-8"))
bird = json.loads((DATA_DIR / "bird_large.json").read_text(encoding="utf-8"))
DBS = ["financial", "toxicology", "codebase_community"]
profiles = {db: json.loads((DATA_DIR / f"{db}_profile.json").read_text(encoding="utf-8")) for db in DBS}
questions = [q for q in bird if q["db_id"] in DBS]

print("GPU:", torch.cuda.get_device_name(0))
print("вопросов для замера:", len(questions))
for v in VARIANTS:
    n = len(json.loads((DATA_DIR / f"train_{v}.json").read_text(encoding="utf-8")))
    print(f"  {v:<16} {n:>4} пар")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Тип весов обязан совпадать с флагом тренера, иначе GradScaler падает на bf16.
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

FENCED = re.compile(r"```(?:sql)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)
STATEMENT = re.compile(r"\b(WITH|SELECT)\b", re.IGNORECASE)


def build_prompt(db, question):
    """Схемы нет ни в одном арме ablation: все они проверяют, чему модель
    научилась из данных, а не что ей подсказали в контексте."""
    system = f"You are a PostgreSQL expert for the database `{db}`. Return only SQL."
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": question}],
        tokenize=False,
        add_generation_prompt=True,
    )


def clean_sql(text):
    """Проза вокруг блока кода уедет в базу как SQL и получит ноль — вырезаем."""
    text = text.strip()
    fenced = FENCED.search(text)
    if fenced:
        text = fenced.group(1).strip()
    start = STATEMENT.search(text)
    if start:
        text = text[start.start() :]
    return text.strip().rstrip(";").strip()


def fresh_model():
    """Каждый вариант учится с нуля: peft правит слои на месте, и переиспользование
    объекта наложило бы второй адаптер поверх первого."""
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    kw = dict(quantization_config=bnb, device_map={"": 0})
    try:
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=COMPUTE_DTYPE, **kw)
    except TypeError:
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=COMPUTE_DTYPE, **kw)
    m.config.use_cache = False
    return m


print("считаем в", COMPUTE_DTYPE)

In [ ]:
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

SUPPORTED = {f.name for f in fields(SFTConfig)}


def to_ds(pairs):
    return Dataset.from_list(
        [{"text": build_prompt(p["db_id"], p["question"]) + p["sql"] + tokenizer.eos_token} for p in pairs]
    )


def train_variant(variant):
    pairs = json.loads((DATA_DIR / f"train_{variant}.json").read_text(encoding="utf-8"))
    model = prepare_model_for_kbit_training(fresh_model())

    use_bf16 = COMPUTE_DTYPE is torch.bfloat16
    kwargs = dict(
        output_dir=str(OUT_DIR / "ckpt" / variant),
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        logging_steps=20,
        save_strategy="no",
        bf16=use_bf16,
        fp16=not use_bf16,
        optim="paged_adamw_8bit",
        dataset_text_field="text",
        report_to="none",
        max_length=512,
    )
    args = SFTConfig(**{k: v for k, v in kwargs.items() if k in SUPPORTED})
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=to_ds(pairs),
        eval_dataset=to_ds(val_pairs),
        peft_config=LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        ),
    )
    trainer.train()
    return trainer


def predict(trainer, variant):
    trainer.model.eval()
    trainer.model.config.use_cache = True
    predictions = {}
    for i, q in enumerate(questions, 1):
        question = f"question: {q['question']}, evidence (may be empty): {q['evidence']}"
        inputs = tokenizer(build_prompt(q["db_id"], question), return_tensors="pt").to(trainer.model.device)
        with torch.no_grad():
            out = trainer.model.generate(
                **inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.eos_token_id
            )
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        predictions[str(q["question_id"])] = clean_sql(text)
        if i % 30 == 0:
            print(f"    {variant}: {i}/{len(questions)}")

    out_path = OUT_DIR / f"query_results_{variant}.json"
    out_path.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
    return out_path


done = {}
for variant in VARIANTS:
    print(f"\n===== {variant} =====")
    trainer = train_variant(variant)
    done[variant] = str(predict(trainer, variant))
    print(f"  пик памяти: {torch.cuda.max_memory_allocated() / 1e9:.2f} ГБ")

    # Иначе следующий вариант не влезет: старая модель держит память.
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

(OUT_DIR / "ablation_files.json").write_text(json.dumps(done, indent=2), encoding="utf-8")
print("\nготово:", json.dumps(done, indent=2))

## Дома

Скачай все `query_results_*.json` и посчитай EX по каждому:

```bash
for v in synth_50 synth_143 synth_347 unfiltered_347 real_143; do
  uv run --env-file .env python bird_evaluate_only.py \
      query_results_$v.json data/bird_large.json
done
```

Что читаем в числах:

- `synth_50 → synth_143 → synth_347` — кривая насыщения. Если EX ещё растёт, синтетики мало.
- `synth_347` против `unfiltered_347` — цена фильтра по исполнению. В нефильтрованном 24% битого SQL.
- `real_143` против `synth_143` — насколько учитель хуже живой разметки, при равном размере.